# Hestia Assignment 8 — Person 2 hosted completion run

Use **Runtime → Change runtime type → Hardware accelerator: CPU**. This keeps the team's frozen generator on **CPU, float32, four PyTorch threads**. A GPU is not used. Run cells in order and keep downloaded evidence: Colab's temporary runtime may disconnect.

This executes **15 small-model matrix requests** and **10 paired scaling requests** with Person 3's unchanged generator and runner. Full retrieved passages, prompts, actual answers and measured latency are preserved. Grading happens after the run; the answer key never enters model inputs.

Upload the matching `hestia-a8-person2-colab.zip` when requested. After completion return the evidence ZIP and this executed notebook. The November 9 run completed successfully; its returned notebook is preserved separately as person2_colab_executed.ipynb. This file starts a new run.

In [ ]:
import sys,subprocess
# Keep Colab's supplied PyTorch/NumPy; pin the same embedding/Transformers stack.
subprocess.check_call([sys.executable,'-m','pip','install','--quiet',
    'sentence-transformers==5.1.2','transformers==4.57.6',
    'huggingface-hub==0.36.2','tokenizers==0.22.2'])
import torch
from packaging.version import Version
assert Version(torch.__version__.split('+')[0])>=Version('2.2'), 'Use a current Colab runtime'
torch.set_num_threads(4)
print('Python:',sys.version.split()[0],'PyTorch:',torch.__version__,'threads:',torch.get_num_threads())

In [ ]:
from pathlib import Path
import hashlib,json,zipfile,os
from google.colab import files
EXPECTED_SHA256='1a936d6b4bdfe3e94a7e3e86af805e917cf7151121c3c747916339ecb6f4acf5'
WORK=Path.cwd()/'hestia_a8_person2'
uploaded=files.upload()
assert len(uploaded)==1, 'Upload only hestia-a8-person2-colab.zip'
name,data=next(iter(uploaded.items()))
assert hashlib.sha256(data).hexdigest()==EXPECTED_SHA256, 'Wrong bundle/version; use the matching ZIP'
WORK.mkdir(parents=True,exist_ok=True)
import io
with zipfile.ZipFile(io.BytesIO(data)) as archive:
    for item in archive.infolist():
        assert (WORK/item.filename).resolve().is_relative_to(WORK.resolve()), 'Unsafe archive path'
    archive.extractall(WORK)
input_manifest=json.loads((WORK/'bundle_manifest.json').read_text())
for path,digest in input_manifest['files'].items():
    assert hashlib.sha256((WORK/path).read_bytes()).hexdigest()==digest, path
sys.path.insert(0,str(WORK))
os.chdir(WORK)
print('Verified',len(input_manifest['files']),'input files; base commit:',input_manifest['base_commit'])

In [ ]:
import csv
from a8.person3_experiments import run
from a8.person2_experiments import read_rows,write_json
from a8.retrieval import fingerprint
from a8.generation import MODEL,REVISION,MAX_NEW_TOKENS
RUN=Path('a8/results/person2/completed-2026-11-09')
RUN.mkdir(parents=True,exist_ok=True)
write_json(RUN/'input_bundle_manifest.json',input_manifest)
print('Generator:',MODEL,REVISION,'float32 CPU; greedy; output cap:',MAX_NEW_TOKENS)
print('Two datasets, fixed small embedding revision, hybrid for scaling, k=3; 15+10 requests total.')

def download_evidence(name='hestia-a8-person2-colab-evidence.zip'):
    target=Path(name)
    with zipfile.ZipFile(target,'w',zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(RUN.rglob('*')):
            if path.is_file(): archive.write(path,str(path))
    files.download(str(target))
    print('Evidence exported:',target.name)

## Matrix: three chunkers × five queries

A completed row is saved immediately. Same-runtime reruns skip matching completed rows. Downloads/index builds/warm-up are excluded from latency. If this cell errors, its `finally` block still offers partial evidence for download; preserve it and the exact error.

In [ ]:
try:
    run('small',RUN/'matrix')
finally:
    download_evidence('hestia-a8-person2-colab-matrix.zip')
print('Matrix responses:',len(read_rows(RUN/'matrix/rows.jsonl')))

## Scaling: small versus full dataset

The same five questions retain their answer evidence in both datasets. Keep the encoder, generator, prompt, decoding, output cap and k=3 unchanged. Only dataset content changes. Both datasets use hybrid chunking.

In [ ]:
try:
    run('small',RUN/'scaling',scaling=True)
finally:
    download_evidence('hestia-a8-person2-colab-scaling.zip')
print('Scaling responses:',len(read_rows(RUN/'scaling/rows.jsonl')))

In [ ]:
from a8.corpus import STRATEGIES,load_queries
collected=[]
for experiment in ('matrix','scaling'):
    raw=read_rows(RUN/experiment/'rows.jsonl')
    datasets=('full',) if experiment=='matrix' else ('small','full')
    strategies=STRATEGIES if experiment=='matrix' else ('hybrid',)
    expected={f"small/{d}/{s}/{q['id']}" for d in datasets for s in strategies for q in load_queries()}
    assert {r['row_id'] for r in raw}==expected, f'Incomplete {experiment}'
    for row in raw:
        assert row['answer'] and row['prompt'] and row['context_truncated'] is False
        assert len(row['retrieved_chunks'])==3
        collected.append((experiment,row))
print('Confirmed: 15 matrix + 10 scaling actual responses with full contexts and E2E timings.')

# This is a template, not a score: all judgment fields remain empty until reviewed.
reviews=[dict(experiment=e,row_id=r['row_id'],evidence_sha256=fingerprint(r),
    matched_facts='',unsupported_count='',relevant_ranks='',notes='PENDING REVIEW') for e,r in collected]
with (RUN/'reviews_template.csv').open('w',newline='',encoding='utf-8') as f:
    writer=csv.DictWriter(f,fieldnames=list(reviews[0]),lineterminator='\n')
    writer.writeheader();writer.writerows(reviews)
answers=[dict(experiment=e,row_id=r['row_id'],dataset=r['dataset'],strategy=r['strategy'],
    query=r['query'],answer=r['answer'],evidence_quote_coverage=r['evidence_quote_coverage'],
    retrieval_ms=r['retrieval_ms'],generation_ms=r['generation_ms'],end_to_end_ms=r['end_to_end_ms'],
    reached_token_cap=r['reached_token_cap'],retrieved_chunks=json.dumps(r['retrieved_chunks'],ensure_ascii=False))
    for e,r in collected]
with (RUN/'unreviewed_answers.csv').open('w',newline='',encoding='utf-8') as f:
    writer=csv.DictWriter(f,fieldnames=list(answers[0]),lineterminator='\n')
    writer.writeheader();writer.writerows(answers)
download_evidence()

## Save and return

Return **hestia-a8-person2-colab-evidence.zip** and **File → Download → Download .ipynb**. The complete run includes 25 actual answers; no quality scores have been invented. We will review required facts/unsupported instructions under Person 3's rubric, inspect relevance/noise, validate all review hashes and export the final 15-row table and scaling findings.

Do not simply rename the review template to a completed assessment. Each review must reflect the saved answer and exact retrieved context. Colab timings are a separate measured environment from the earlier Mac retrieval runs.